# Talos: a short introduction

Use Python 3.11 or later with the Talos tensorflow and plots extras installed. These short demonstration budgets can be increased for a research run.


# Predicting Breast Cancer with Keras

### A Very Short Introduction to Hyperparameter Optimization with Talos

The goal of a Talos experiment, is to find a set of suitable hyperparameters for Keras model. In order to do this, you need to have three things: 

- Keras model 
- Talos hyperparameter dictionary
- Talos experiment configuration

Below we will briefly overview each.

As a model, any Keras model will do. This example uses the real Wisconsin breast cancer dataset bundled with scikit-learn. No download is required. The two-epoch defaults keep the demonstration short; increase them for a research experiment.

In [ ]:
from sklearn.datasets import load_breast_cancer
x, y = load_breast_cancer(return_X_y=True)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Normalization

def breast_cancer():
    normalization = Normalization()
    normalization.adapt(x)
    model = Sequential([Input(shape=(x.shape[1],)), normalization,
                        Dense(12, activation='relu'), Dense(1, activation='sigmoid')])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(x, y, epochs=2, batch_size=32, verbose=0)
    return model

baseline = breast_cancer()

### Talos Hyperparameter Dictionary

Let's prepare for an experiment where we will optimize against three common attributes:

- neurons on the first layer
- activations
- batch_sizes

In [ ]:
p = {'first_neuron': [12, 24],
     'activation': ['relu', 'elu'],
     'batch_size': [32], 'epochs': [2]}

### Configuring the Keras Model for Talos

In order to prepare a Keras model for a Talos experiment, we need to do four things:

- add input parameters to the function
- replace the hyperparameter inputs with references to params dictionary
- make sure model.fit() stores the history object
- modify the output of the model

These steps are always the same.

In [ ]:
def breast_cancer(x_train, y_train, x_val, y_val, params):
    # Fit preprocessing on the training partition only and save it with the model.
    normalization = Normalization()
    normalization.adapt(x_train)
    model = Sequential([Input(shape=(x_train.shape[1],)), normalization,
                        Dense(params['first_neuron'], activation=params['activation']),
                        Dense(1, activation='sigmoid')])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=params['epochs'], batch_size=params['batch_size'], verbose=0)
    return history, model

That's it, there is nothing more to it. A more complicated experiment would just entail more of the same in terms of the way the params dictionary references are made. Otherwise the changes would always be exactly the same.

### Talos Experiment

The Talos experiment is performed through the Scan() command. In case you don't have Talos installed already, you can do that now.

In [ ]:
import talos

While many configurations are possible, the required inputs are x, y, params (the dictionary p), and model (the breast_cancer function).

In [ ]:
t = talos.Scan(x=x, y=y, params=p, model=breast_cancer, experiment_name='breast_cancer_intro', seed=17, disable_progress_bar=True)
assert len(t.data) == 4